# Chapter 2: Lab 1 - Hands-on lab: Retrieving fundamental ratios for Apple

## Notebook Overview

This hands-on lab implements the **tool-use pattern**: an OpenAI Agents SDK agent, running on `gpt-4o-mini`, is asked a question about Apple's fundamentals and answers it by *calling a tool* that fetches the numbers live from a financial-data API, rather than relying on the model's static training knowledge. It is the simplest and most foundational design pattern, and it also exposes one of the key limitations an LLM has on its own: its knowledge has a cutoff date.

The setup installs two packages — `openai-agents` (the SDK that provides `Agent`, `Runner` and the `@function_tool` decorator) and `financetoolkit` (the library that wraps the Financial Modeling Prep API). Next, the notebook loads two keys from the `.env` file at the repository root: `OPENAI_API_KEY`, which the SDK needs, and `FINANCIAL_MODELING_PREP_API_KEY`, which is used to call the fundamentals service.

The tool itself is a function decorated with `@function_tool` called `get_company_fundamentals(ticker)`. Internally it builds a `Toolkit([ticker], api_key=FINANCIAL_MODELING_PREP_API_KEY, start_date="2022-01-01")` and calls `ratios.collect_all_ratios()`, which downloads Apple's income statement, balance sheet and cash-flow statement since 2022 and derives dozens of ratios from them. The function then keeps only the rows that matter for this lab: the profitability ratios (return on equity, return on assets, gross / operating / net margins), the valuation ratios (price-to-earnings, price-to-book, price-to-cash-flow, EV/EBITDA, and so on) and the liquidity ratios (current, quick and cash). What comes back is a pandas DataFrame indexed by ratio name.

The **fundamental retrieval agent** is then defined: a `gpt-4o-mini` agent whose only instruction is to call the fundamentals tool for a given ticker and output the requested set of ratios. It has no other tools, so the whole point is that the model must decide, via function calling, when and how to invoke the data source.

The first run is deliberately instructive: the prompt asks the agent to list three liquidity ratios and give Apple's values for the *most recent fiscal year*. Because the model's knowledge was frozen around 2024, it answers with **2023** data — it genuinely believes 2023 is the latest year. This is a clean demonstration of the knowledge-cutoff problem: the LLM cannot know today's date or which fiscal year reporting has reached. The second run fixes this by adding one sentence to the prompt — *"knowing that we are in 2026"* — and now the agent correctly retrieves the **2025** figures. In both cases the tool returned real, live data; only the model's *choice of year* was imperfect.

The lab closes by running the Toolkit directly, bypassing the agent entirely, so you can see exactly what the tool hands over: the aggregated `ratios` DataFrame, followed by categorical views such as the profitability ratios, to confirm the tool's output shape before the LLM ever touches it.

**The point of the lab:** tool use means the LLM decides when to call external functions, and the tool provides live, factual data the model could never know from training, so answers stay current and accurate. But the model still needs framing — like the current year — to ask the right question. This primitive, a single agent with a single tool, is the building block on which every later lab in the chapter stacks more complexity.


In this hands-on lab, we implement an agent using the **tool use pattern**. This agent will be requested to retrieve the liquidity ratios of Apple.

We use OpenAI Agents SDK with GPT-4o-mini as the underlying LLM.

The tool used in this hands-on lab "`get_company_fundamentals`" will invoke Finance Toolkit that is using the Financial Modeling Prep API to collect various sets of fundamental ratios:

https://www.jeroenbouma.com/projects/financetoolkit

https://site.financialmodelingprep.com/

You need to create an API Key on https://site.financialmodelingprep.com/: create an account and then create an API Key visible in your dashboard.


## Install Packages

In [1]:
%pip install openai-agents -q
%pip install financetoolkit -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Specify the API Keys:

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

FINANCIAL_MODELING_PREP_API_KEY = os.getenv("FINANCIAL_MODELING_PREP_API_KEY")

In [3]:
from financetoolkit import Toolkit

In [4]:
from agents import Agent, Runner, WebSearchTool, trace, ItemHelpers, TResponseInputItem, function_tool
import pandas as pd

## Specify the tool

The tool `get_company_fundamentals` retrieves a stock’s fundamental ratios, including profitability, valuation, and liquidity metrics.


In [5]:
@function_tool
def get_company_fundamentals(ticker: str) -> dict[str:pd.DataFrame]:
  """ Get profitability, valuation and liquidity ratios for a given ticker. """

  companies = Toolkit(
      [ticker], api_key=FINANCIAL_MODELING_PREP_API_KEY, start_date="2022-01-01"
  )
  ratios = companies.ratios.collect_all_ratios()

  # Keep only profitability, valuation and liquidity ratios
  profitability_ratios_names = ['Return on Equity','Return on Assets','Gross Margin','Operating Margin','Net Profit Margin']
  valuation_ratios_names = ['Price-to-Earnings',  "Price-to-Book", "Price-to-Cash-Flow", "Price-to-Free-Cash-Flow", 'EV-to-EBITDA','EV-to-Sales']
  liquidity_ratios_names = ["Current Ratio","Quick Ratio","Cash Ratio","Working Capital","Operating Cash Flow Ratio","Operating Cash Flow to Sales Ratio","Short Term Coverage Ratio"]


  return {
      "profitability_ratios":ratios.loc[ratios.index.intersection(profitability_ratios_names)],
      "valuation_ratios":ratios.loc[ratios.index.intersection(valuation_ratios_names)],
      "liquidity_ratios": ratios.loc[ratios.index.intersection(liquidity_ratios_names)]}


This function defines how to extract key fundamental ratios for a given company ticker.

It connects to a financial data provider, collects all available financial ratios starting from 2022, and then filters them into three categories: profitability, valuation, and liquidity.

* The **profitability** ratios measure how efficiently a company generates profit relative to its revenue, assets, or equity. They indicate the firm’s ability to create value from its operations (e.g., Return on Equity (ROE), Net Profit Margin).  
* The **valuation** ratios assess how the market values the company compared to its earnings, cash flows, or assets. They help determine whether a stock appears expensive or undervalued (e.g., Price-to-Earnings (PE), EV-to-EBITDA).
* The **liquidity** ratios evaluate a company’s ability to meet its short-term obligations. They indicate whether the firm has sufficient liquid resources to cover immediate liabilities (e.g., Current Ratio (CR), Quick Ratio(QR)).

Instead of returning the full dataset, the function selectively returns only the most important ratios for our use case, such as Return on Equity (ROE), Price-to-Earnings (PE), and Current Ratio (CR), organized into structured output.

## Specify the Fundamental retrieval Agent

 The `fundamental_retrieval_agent` is instructed to invoke the fundamentals tool when needed and return the desired set of financial ratios for the specified ticker:


In [6]:
MODEL = "gpt-4o-mini"
fundamental_retrieval_agent = Agent(
    name="Fundamental Retrieval Agent",
    model=MODEL,
    instructions="""
You are a Fundamental Retrieval Agent.

Tasks:
1) Call the fundamentals tool for the specified ticker.
2) Output the desired set of ratios.

""",
    tools=[get_company_fundamentals],
)

## Run the agent

In the following prompt I ask the agent (LLM behind it) to retrieve ratios for the most recent fiscal year: (looks which year it retrieves ⬇ )

In [7]:

fundamental_results = await Runner.run(fundamental_retrieval_agent,
                                       """Proceed step by step:
                                       1- List 3 liquidity ratios and their signification.
                                       2- Provide Apple’s values for the three listed liquidity ratios for the most recent fiscal year.
                                       """)
print(fundamental_results.final_output)


2026-09-23 00:00:05 - financetoolkit - INFO - Obtaining financial statements
2026-09-23 00:00:05 - financetoolkit - INFO - Obtaining balance data for 1 ticker(s)
2026-09-23 00:00:06 - financetoolkit - INFO - Obtaining income data for 1 ticker(s)
2026-09-23 00:00:07 - financetoolkit - INFO - Obtaining cashflow data for 1 ticker(s)
2026-09-23 00:00:09 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
c:\Users\Riemann\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
2026-09-23 00:00:16 - financetoolkit - INFO - Obtaining historical data for 2 ticker(s)


### Step 2: Apple's Liquidity Ratios for the Most Recent Fiscal Year (2023)

1. **Current Ratio**: 0.988
   - Indicates that Apple has slightly less than 1 in current assets for every dollar of current liabilities.

2. **Quick Ratio**: 0.6267
   - Suggests Apple has approximately 62.67 cents in liquid assets for every dollar of current liabilities, excluding inventory.

3. **Cash Ratio**: 0.4236
   - Implies that Apple has 42.36 cents in cash and cash equivalents for every dollar of current liabilities.

These ratios provide insight into Apple's liquidity position and its ability to cover short-term obligations.


The agent retrieves data for 2023 because the LLM relies on its knowledge cutoff (2024) to infer what it considers the most recent completed year.

In this run, I modified slighlty the prompt to sepcify that we are in 2026:

In [8]:

fundamental_results = await Runner.run(fundamental_retrieval_agent,
                                       """Proceed step by step:
                                       1- List 3 liquidity ratios and explain their significance.
                                       2- Provide Apple’s values for the three listed liquidity ratios for the most recent fiscal year, knowing that we are in 2026.
                                       """)
print(fundamental_results.final_output)

2026-09-23 00:01:11 - financetoolkit - INFO - Obtaining financial statements
2026-09-23 00:01:11 - financetoolkit - INFO - Obtaining balance data for 1 ticker(s)
2026-09-23 00:01:13 - financetoolkit - INFO - Obtaining income data for 1 ticker(s)
2026-09-23 00:01:14 - financetoolkit - INFO - Obtaining cashflow data for 1 ticker(s)
2026-09-23 00:01:16 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
c:\Users\Riemann\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
2026-09-23 00:01:17 - financetoolkit - INFO - Obtaining historical data for 2 ticker(s)


### 1. Three Liquidity Ratios and Their Significance

1. **Current Ratio**
   - **Definition:** The current ratio is calculated by dividing current assets by current liabilities. It measures a company's ability to cover its short-term obligations with its short-term assets.
   - **Significance:** A ratio above 1 indicates that the company has more current assets than current liabilities, suggesting a good short-term financial health. A ratio below 1 could indicate potential liquidity problems.

2. **Quick Ratio (Acid-Test Ratio)**
   - **Definition:** The quick ratio is similar to the current ratio but excludes inventory from current assets. It is calculated as (Current Assets - Inventories) / Current Liabilities.
   - **Significance:** This ratio provides a stricter measure of liquidity since it considers only the most liquid assets. A quick ratio above 1 is generally favorable, indicating that a company can meet its short-term obligations without relying on selling inventory.

3. **C

This time, the agent provides the correct information, it retrieves 2025 fiscal year data.

**Conclusion**:
This result illustrates the strength of the tool use pattern: the agent is able to retrieve up-to-date, structured financial data from an external source and present it in a clear, consistent format. It also provides insights for each ratio, along with an explanation of what is generally considered a healthy value.

Run the toolkit directly:

In [9]:
companies = Toolkit(
      ["AAPL"], api_key=FINANCIAL_MODELING_PREP_API_KEY, start_date="2022-01-01"
  )
ratios = companies.ratios.collect_all_ratios()

2026-09-23 00:02:03 - financetoolkit - INFO - Obtaining financial statements
2026-09-23 00:02:03 - financetoolkit - INFO - Obtaining balance data for 1 ticker(s)
2026-09-23 00:02:03 - financetoolkit - INFO - Obtaining balance data for 1 ticker(s)
2026-09-23 00:02:04 - financetoolkit - INFO - Obtaining income data for 1 ticker(s)
2026-09-23 00:02:05 - financetoolkit - INFO - Obtaining cashflow data for 1 ticker(s)
2026-09-23 00:02:07 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
c:\Users\Riemann\AppData\Local\Programs\Python\Python312\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
2026-09-23 00:02:08 - financetoolkit - INFO - Obtaining historical data for 2 ticker(s)


In [10]:
liquidity_ratios_3 = [
    "Current Ratio",
    "Quick Ratio",
    "Cash Ratio",
]
ratios.loc[ratios.index.intersection(liquidity_ratios_3)]

,2022,2023,2024,2025
Current Ratio,0.8794,0.988,0.8673,0.8933
Quick Ratio,0.4967,0.6267,0.5589,0.5704
Cash Ratio,0.3137,0.4236,0.3695,0.3302
